In [1]:
import os, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
torch.cuda.empty_cache()

# pretrained model

In [2]:
from model_skingpt4 import *

/home/jq2uw/miniconda3/envs/skingpt4/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model, vis_processor, chat = init_chat()
sum(p.numel() for p in model.parameters() if p.requires_grad), sum(p.numel() for p in model.parameters())

Initializing Chat
Loading VIT


/home/jq2uw/miniconda3/envs/skingpt4/lib/python3.9/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading VIT Done
Loading Q-Former
Loading Q-Former Done
Loading LLM tokenizer
Loading LLM model


Loading checkpoint shards: 100%|█████████████████████████████████████████████████| 3/3 [00:06<00:00,  2.25s/it]


Loading LLM Done
Load 2 training prompts
Prompt Example 
###Human: <Img><ImageHere></Img> What's wrong with my skin? ###Assistant: 
Load BLIP2-LLM Checkpoint: /scratch/jq2uw/edit-skingpt4/model_skingpt4/weights/skingpt4_llama2_13bchat_base_pretrain_stage2.pth
Initialization Finished


(3937280, 14110861184)

# data

In [4]:
from data_utils import *

In [5]:
df = process_tabular("./data")
train_df, val_df, test_df = split_df(df)
train_dataset = MIDASDataset(train_df, "./data")
val_dataset = MIDASDataset(val_df, "./data")
test_dataset = MIDASDataset(test_df, "./data")

id_patient: 733
id_filename: 3416
midas_path: 17
midas_path: {'malignant-bcc': 608, 'benign-melanocytic nevus': 578, 'nan': 497, 'benign-other': 421, 'benign-seborrheic keratosis': 242, 'malignant-melanoma': 238, 'malignant-scc': 203, 'malignant-ak': 191, 'malignant-sccis': 165, 'other-melanocytic lesion, possible re-excision (severe, spitz, aimp)': 109, 'benign-dermatofibroma': 51, 'other-non-neoplastic, inflammatory, infectious': 39, 'benign-hemangioma': 30, 'benign-fibrous papule': 18, 'malignant-other': 14, 'melanocytic tumor, possible re-excision (severe, spitz, aimp)': 6, 'unknown': 6}
y16_description: 16
y16: 16
y16: {'Basal Cell Carcinoma': 608, 'Melanocytic Nevus': 578, 'Other Benign': 421, 'Seborrheic Keratosis': 242, 'Melanoma': 238, 'Squamous Cell Carcinoma': 203, 'Actinic Keratosis': 191, 'Squamous Cell Carcinoma In Situ': 165, 'Melanocytic Lesion': 109, 'Dermatofibroma': 51, 'Non-neoplastic': 39, 'Hemangioma': 30, 'Fibrous Papule': 18, 'Other Malignant': 14, 'Melanocytic 

## finetune

In [6]:
from finetune_utils import *

In [ ]:
# vis_processor is already initialized from run/init.py
target = 'y3'
train_ds_ft = MIDASFTSkGPT4Dataset(train_dataset, vis_processor, target)
val_ds_ft   = MIDASFTSkGPT4Dataset(val_dataset,   vis_processor, target)
train_loader = train_ds_ft.get_loader(batch_size=2, shuffle=True, num_workers=2)
val_loader   = val_ds_ft.get_loader(batch_size=2, shuffle=False, num_workers=2)

model = load_model_weights(model, "./model_skingpt4/weights/finetune_llama.pth")
model = finetune(model, train_loader, val_loader, n_epochs=10, retrain=True,
        lr=1e-4, weight_decay=0.0, ckpt_path="./model_skingpt4/weights/finetune_llama.pth")

epoch 1/5  train_loss=0.1988  val_loss=0.2528
epoch 2/5  train_loss=0.1816  val_loss=0.2608
epoch 3/5  train_loss=0.1704  val_loss=0.2846

KeyboardInterrupt: saved checkpoint to ./model_skingpt4/weights/finetune_llama.pth


## eval

In [8]:
from eval_utils import *

In [13]:
# one example
i = 16
image = test_dataset[i]['image']
print(f"ground truth: {test_dataset[i]['y']['y3']}")
print("-" * 50)
print("Pretrained model")
model = load_model_weights(model, "./model_skingpt4/weights/skingpt4_llama2_13bchat_base_pretrain_stage2.pth")
resp = chat_with_image(chat, image, "Is the lesion malignant or benign, or unknown?", temperature=0.01)
print(resp)
print("-" * 50)
print("Finetuned model")
model = load_model_weights(model, "./model_skingpt4/weights/finetune_llama.pth")
resp = chat_with_image(chat, image, "Is the lesion malignant or benign, or unknown?", temperature=0.01)
print(resp)

ground truth: malignant
--------------------------------------------------
Pretrained model
The image shows a close up view of a lesion on the skin, with red, swollen, and scaly edges. The lesion appears to be around 1-2 centimeters in diameter, with a rough texture and some small scabs on the surface. There are also some small blood vessels visible in the center of the lesion, which may indicate inflammation or infection. Without more information or context, it's difficult to determine whether the lesion is malignant or benign, or if it's caused by an unknown condition or allergy.
--------------------------------------------------
Finetuned model
malignant


In [14]:
from torch.utils.data import Subset

subset = Subset(train_dataset, range(0, 200))

res = eval_ft_skingpt4(chat, subset, 
                       temperature=0.01, target="y3", 
                       question="Is the lesion malignant or benign, or other?")
res

100%|████████████████████████████████████████████████████████████████████████| 200/200 [01:24<00:00,  2.37it/s]


{'accuracy': 0.795,
 'precision': 0.7911967891704426,
 'recall': 0.7664244921564375,
 'f1': 0.7727615869042652,
 'report': '              precision    recall  f1-score   support\n\n      benign       0.78      0.58      0.67        53\n   malignant       0.80      0.93      0.86        87\n       other       0.80      0.78      0.79        60\n\n    accuracy                           0.80       200\n   macro avg       0.79      0.77      0.77       200\nweighted avg       0.79      0.80      0.79       200\n',
 'confusion': array([[31, 11, 11],
        [ 5, 81,  1],
        [ 4,  9, 47]])}

In [15]:
from torch.utils.data import Subset

subset = Subset(test_dataset, range(0, 200))

res = eval_ft_skingpt4(chat, subset, 
                       temperature=0.1, target="y3", 
                       question="Is the lesion malignant or benign, or other?")
res

100%|████████████████████████████████████████████████████████████████████████| 200/200 [01:25<00:00,  2.34it/s]


{'accuracy': 0.58,
 'precision': 0.4976360189118487,
 'recall': 0.5056140350877193,
 'f1': 0.49830268787730625,
 'report': '              precision    recall  f1-score   support\n\n      benign       0.37      0.28      0.32        50\n   malignant       0.71      0.74      0.72       114\n       other       0.42      0.50      0.46        36\n\n    accuracy                           0.58       200\n   macro avg       0.50      0.51      0.50       200\nweighted avg       0.57      0.58      0.57       200\n',
 'confusion': array([[14, 24, 12],
        [17, 84, 13],
        [ 7, 11, 18]])}